# ⚠️ Google Colab Setup (Read First)
To run this lab successfully, you **must** enable the Free GPU.

1. In the menu bar at the top, click **Runtime** > **Change runtime type**.
2. Under "Hardware accelerator", select **T4 GPU** and click **Save**.
3. *(Note: You must be signed in with a free Google account to use Colab)*

> **💡 Solution Available:** Try completing this lab on your own first. When you're done, open **`C2_HOL1_Datasets_and_Fine_Tuning_Solution.ipynb`** from the file browser (left panel) in the same folder to compare your outputs with the expected exemplar.


# C2-HOL1: Fine-Tune a Sentiment Classifier for NovaPay

| Field | Detail |
|-------|--------|
| **Course** | Building AI Apps with HF Spaces and Gradio |
| **Module** | M1: Working with HF Datasets and Fine-Tuning |
| **Complexity** | Complex |
| **Duration** | 20 minutes |
| **Environment** | Google Colab free T4 GPU |

---

## Learning Objective

**LO1 (Apply):** Load and preprocess HF Hub datasets, fine-tune a pre-trained model with the Trainer API, compute evaluation metrics, and push the result to the Hub with a model card.

---

## Scenario

It is **Monday afternoon at NovaPay**. Sara (VP of Engineering) has decided the off-the-shelf sentiment classifier isn't good enough for NovaPay's financial customer messages. She wants a model **fine-tuned for the domain**.

You will fine-tune DistilBERT on the IMDB dataset as a proof-of-concept, evaluate it with real metrics, and **publish it to the HF Hub** — creating an artifact that the rest of the course builds on.

---

## Prerequisites

- Google account (for Colab)
- Free Hugging Face account with a **write access token** (for `push_to_hub`)
- Create your token at: https://huggingface.co/settings/tokens

---

## Setup

Run the cell below to install required packages and verify GPU availability.

In [ ]:
# Setup — install required packages
!pip install -q transformers datasets evaluate accelerate huggingface_hub
# Imports
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected! Fine-tuning will be very slow.")
    print("   Go to Runtime → Change runtime type → T4 GPU")

print("✅ Setup complete!")

---

## Task 1: Load and Explore the IMDB Dataset (3 min)

Load the IMDB dataset from the Hub. Inspect the splits, features, and label distribution.

In [ ]:
# TODO: Load the IMDB dataset from the Hub
dataset = load_dataset("stanfordnlp/imdb")

# Explore the dataset
print(f"=== IMDB Dataset ===")
print(f"  Splits: {list(dataset.keys())}")
print(f"  Training examples:   {len(dataset['train'])}")
print(f"  Test examples:       {len(dataset['test'])}")
print(f"  Features:            {dataset['train'].features}")

In [ ]:
# Inspect label distribution
import collections
label_counts = collections.Counter(dataset['train']['label'])
print(f"Label distribution (train): {dict(label_counts)}")
print(f"  0 = Negative, 1 = Positive")

# Preview a sample
print(f"\n=== Sample Review ===")
print(f"  Text: {dataset['train'][0]['text'][:200]}...")
print(f"  Label: {dataset['train'][0]['label']} ({'Positive' if dataset['train'][0]['label'] == 1 else 'Negative'})")

#### ✅ Verification
You should see 25,000 training examples and 25,000 test examples, with roughly equal positive/negative distribution.

---

## Task 2: Tokenize with map() and Dynamic Padding (4 min)

Load the DistilBERT tokenizer. Tokenize the full dataset using `map(batched=True)` for efficiency. Configure `DataCollatorWithPadding` for dynamic padding during training.

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# TODO: Define the tokenization function
# Hint: Use truncation=True and max_length=256
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

# TODO: Apply tokenization to the full dataset using map(batched=True)
tokenized = dataset.map(tokenize_fn, batched=True)

print(f"✅ Tokenization complete!")
print(f"  Columns: {tokenized['train'].column_names}")
print(f"  Sample input_ids length: {len(tokenized['train'][0]['input_ids'])}")

In [ ]:
# TODO: Configure the DataCollator for dynamic padding
# Dynamic padding = pad to the longest sequence in each batch, not globally
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ DataCollatorWithPadding configured!")
print("💡 Dynamic padding is more efficient than padding everything to max_length")

In [ ]:
# Create smaller subsets for time-constrained training
# In production, you would use the full 25,000-example dataset
small_train = tokenized["train"].shuffle(seed=42).select(range(5000))
small_test = tokenized["test"].shuffle(seed=42).select(range(1000))

print(f"Training subset: {len(small_train)} examples")
print(f"Test subset:     {len(small_test)} examples")
print(f"\n💡 Using a subset for the lab (~3-4 min training). Production would use all 25,000.")

#### ✅ Verification
- Tokenized dataset should have new columns: `input_ids` and `attention_mask`
- Subsets should be 5,000 (train) and 1,000 (test) examples

---

## Task 3: Configure TrainingArguments and Fine-Tune (6 min)

Load `AutoModelForSequenceClassification` with 2 labels. Configure training hyperparameters and launch training.

In [ ]:
# TODO: Load the pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2
)

print(f"✅ Model loaded!")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TODO: Define the compute_metrics function for evaluation
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

print("✅ Metrics function defined!")

In [ ]:
# TODO: Configure TrainingArguments
training_args = TrainingArguments(
    output_dir="novapay-sentiment",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"  # Disable wandb etc.
)

print("✅ TrainingArguments configured!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
# TODO: Create the Trainer and launch training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("🚀 Starting training... (this takes ~3-4 minutes on T4 GPU)")
train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"  Training loss: {train_result.training_loss:.4f}")

#### ✅ Verification
Training should complete in ~3-4 minutes on a T4 GPU. You should see decreasing training loss across the 2 epochs.

---

## Task 4: Evaluate with Accuracy and F1 (3 min)

Compute accuracy and F1 on the test set. Interpret: is the model good enough for a proof-of-concept?

In [ ]:
# TODO: Run evaluation on the test set
eval_results = trainer.evaluate()

print(f"=== Evaluation Results ===")
print(f"  Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_results['eval_f1']:.4f}")
print(f"  Eval Loss: {eval_results['eval_loss']:.4f}")

# Interpret the results
if eval_results['eval_accuracy'] > 0.85:
    print(f"\n✅ Strong results! This proof-of-concept shows fine-tuning is viable.")
else:
    print(f"\n⚠️  Results are below target. Consider more training data or epochs.")

print(f"\n💡 Expected: accuracy ~0.87-0.90, F1 ~0.87-0.90 on this subset.")
print(f"   Production training on full 25K examples would likely improve these scores.")

#### ✅ Verification
Expected results: accuracy ~0.87-0.90 and F1 ~0.87-0.90. These confirm the model has learned meaningful patterns from the training data.

---

## Task 5: Push to Hub and Complete Model Card (4 min)

Authenticate with the HF Hub, push the fine-tuned model, and update the model card.

### Prerequisites
You need a free HF account with a **write token**. Create one at: https://huggingface.co/settings/tokens

In [ ]:
# Authenticate with the HF Hub
# 👇 Paste your HF token below (get one at https://huggingface.co/settings/tokens)
HF_TOKEN = "paste-your-token-here"


In [ ]:
# TODO: Push the fine-tuned model to the Hub
# This creates a new model repository on your HF profile
trainer.push_to_hub("novapay-sentiment-distilbert")

print("✅ Model pushed to the Hub!")
print("📋 Your model URL: https://huggingface.co/<your-username>/novapay-sentiment-distilbert")

In [ ]:
# Update the model card with key information
model_card_content = """
---
license: apache-2.0
tags:
  - text-classification
  - sentiment-analysis
  - novapay
datasets:
  - imdb
metrics:
  - accuracy
  - f1
---

# NovaPay Sentiment Classifier (DistilBERT)

## Intended Use
Fine-tuned for sentiment classification of NovaPay customer support messages.
This is a proof-of-concept model trained on the IMDB dataset.

## Training Data
- **Dataset:** IMDB (5,000 example subset)
- **Base model:** distilbert/distilbert-base-uncased
- **Epochs:** 2
- **Learning rate:** 2e-5

## Evaluation Results
- **Accuracy:** ~0.87-0.90
- **F1 Score:** ~0.87-0.90

## Limitations
- Trained on IMDB movie reviews, NOT actual financial customer messages
- May not generalize well to financial domain-specific language
- Proof-of-concept only — not production-ready
"""

print("=== Model Card ===")
print(model_card_content)
print("\n💡 Copy this content to update your model's README.md on the Hub.")

#### ✅ Final Verification

1. ✅ Your model should be live at `https://huggingface.co/<your-username>/novapay-sentiment-distilbert`
2. ✅ The model card should document intended use, training data, evaluation results, and limitations
3. ✅ Save the URL — you'll need it for C2-HOL2 (building the Gradio app)

---

## Key Takeaways

In this lab, you:
1. **Loaded and explored** the IMDB dataset from the HF Hub
2. **Tokenized efficiently** using `map(batched=True)` and dynamic padding
3. **Fine-tuned DistilBERT** with the Trainer API on a T4 GPU
4. **Evaluated with real metrics** — accuracy and F1 score
5. **Published to the HF Hub** — creating a persistent, reusable artifact

**Next up:** Tuesday (C2-HOL2) — you'll wrap this model in a Gradio app for stakeholder demos.

---
*© NovaPay — Course 2, Day 1 (Monday)*